In [2]:
from langgraph.graph import  StateGraph ,START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
llm = ChatOpenAI()

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):
    prompt = f"Generate a joke on the topic: {state['topic']}"
    response = llm.invoke(prompt).content

    return {'joke':response}
    

In [7]:
def explanation_joke(state:JokeState):
    prompt = f"write an explanation for the joke: {state['joke']}"
    response = llm.invoke(prompt).content
    return {'explanation':response}

In [13]:
# defining the graph

graph = StateGraph(JokeState)

# add nodes to the graph

graph.add_node('generate_joke', generate_joke)
graph.add_node('explanation_joke', explanation_joke)

# add the edge connection
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','explanation_joke')
graph.add_edge('explanation_joke',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [14]:
config1 = {"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'Football'}, config=config1)


{'topic': 'Football',
 'joke': 'Why did the football coach go to the bank? \n\nTo get his quarterback!',
 'explanation': 'This joke is a play on words. In football, the position of quarterback is crucial for a team\'s success. The coach went to the bank not to get money, but rather to "get" (as in recruit or acquire) his quarterback player. It\'s a silly and light-hearted joke that uses wordplay to humorously suggest that the coach went to the bank for an unconventional reason.'}

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach go to the bank? \n\nTo get his quarterback!', 'explanation': 'This joke is a play on words. In football, the position of quarterback is crucial for a team\'s success. The coach went to the bank not to get money, but rather to "get" (as in recruit or acquire) his quarterback player. It\'s a silly and light-hearted joke that uses wordplay to humorously suggest that the coach went to the bank for an unconventional reason.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c219f-a89c-6abc-8002-a345f6cc96d5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-07T06:40:25.850130+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c219f-8e56-61ac-8001-37423567d9c1'}}, tasks=(), interrupts=())

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach go to the bank? \n\nTo get his quarterback!', 'explanation': 'This joke is a play on words. In football, the position of quarterback is crucial for a team\'s success. The coach went to the bank not to get money, but rather to "get" (as in recruit or acquire) his quarterback player. It\'s a silly and light-hearted joke that uses wordplay to humorously suggest that the coach went to the bank for an unconventional reason.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c219f-a89c-6abc-8002-a345f6cc96d5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-07T06:40:25.850130+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c219f-8e56-61ac-8001-37423567d9c1'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football coach go to the bank? \n\nTo g

In [18]:
config2 = {"configurable":{"thread_id":"2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team go to the bank? \n\nTo improve their bowling average!',
 'explanation': 'This joke plays on the double meaning of "bowling." In cricket, "bowling" refers to the action of a player throwing the ball towards the batsman. At the same time, "bowling average" is a statistical term in cricket that represents the total runs scored against a bowler divided by the number of wickets they have taken. By going to the bank, the cricket team is jokingly implying that they are trying to improve their financial situation by improving their bowling average.'}

In [20]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bank? \n\nTo improve their bowling average!', 'explanation': 'This joke plays on the double meaning of "bowling." In cricket, "bowling" refers to the action of a player throwing the ball towards the batsman. At the same time, "bowling average" is a statistical term in cricket that represents the total runs scored against a bowler divided by the number of wickets they have taken. By going to the bank, the cricket team is jokingly implying that they are trying to improve their financial situation by improving their bowling average.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21be-2e83-6dc7-8002-a9e62e2f1fc5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-07T06:54:05.197255+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21be-1266-6cfb-8001-4d6d2deafd88'}}, tasks=(), i

In [21]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bank? \n\nTo improve their bowling average!', 'explanation': 'This joke plays on the double meaning of "bowling." In cricket, "bowling" refers to the action of a player throwing the ball towards the batsman. At the same time, "bowling average" is a statistical term in cricket that represents the total runs scored against a bowler divided by the number of wickets they have taken. By going to the bank, the cricket team is jokingly implying that they are trying to improve their financial situation by improving their bowling average.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21be-2e83-6dc7-8002-a9e62e2f1fc5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-07T06:54:05.197255+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21be-1266-6cfb-8001-4d6d2deafd88'}}, tasks=(), 

In [22]:
# Time Travel
workflow.get_state({"configurable":{"thread_id":"1","checkpoint_id":"1f1c21bd-efd5-6ade-8000-6a740d69efea"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1c21bd-efd5-6ade-8000-6a740d69efea'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [25]:

workflow.invoke(None,{"configurable":{"thread_id":"2","checkpoint_id":"1f1c21bd-efd3-63da-bfff-55ba25f2da34"}})

{'topic': 'Cricket',
 'joke': 'Why did the cricket team go to the bank? \nTo get their bowlers out of debt!',
 'explanation': 'This joke plays on the double meaning of the word "bowlers." In cricket, a "bowler" is a player who delivers the ball to the batsman. However, "bowlers" can also refer to a type of hat. In this joke, the cricket team goes to the bank to get their bowlers (hats) out of debt, implying that their bowlers (players) are in financial trouble.'}

In [27]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bank? \nTo get their bowlers out of debt!', 'explanation': 'This joke plays on the double meaning of the word "bowlers." In cricket, a "bowler" is a player who delivers the ball to the batsman. However, "bowlers" can also refer to a type of hat. In this joke, the cricket team goes to the bank to get their bowlers (hats) out of debt, implying that their bowlers (players) are in financial trouble.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21eb-ff12-6c0b-8003-14405f18d548'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-07T07:14:35.025716+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21eb-dfd5-6010-8002-21b140fd532d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bank? \nTo get their bowlers out o

In [28]:
## Update the state of a thread

In [30]:
workflow.update_state({"configurable":{"thread_id":"2","checkpoint_id":"1f1c21bd-efd5-6ade-8000-6a740d69efea","checkpoint_ns":""}}, {'topic':'Hockey'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1c21f2-eb6a-63a4-8001-054b0e7eda9a'}}

In [31]:
list(workflow.get_state_history({"configurable":{"thread_id":"2"}}))

[StateSnapshot(values={'topic': 'Hockey'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21f2-eb6a-63a4-8001-054b0e7eda9a'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-10-07T07:17:40.869213+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c21bd-efd5-6ade-8000-6a740d69efea'}}, tasks=(PregelTask(id='34baa015-d9fa-0a67-9a90-9f22a8a79caa', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team go to the bank? \nTo get their bowlers out of debt!', 'explanation': 'This joke plays on the double meaning of the word "bowlers." In cricket, a "bowler" is a player who delivers the ball to the batsman. However, "bowlers" can also refer to a type of hat. In this joke, the cricket team goes to the bank